# FINC3012 AUD/USD Call Butterfly PnL Dashboard

This notebook tracks the PnL of your **CME AUD/USD call butterfly** around the RBA event.

Strategy:

- Buy 1x 0.7200 call
- Sell 2x 0.7275 calls
- Buy 1x 0.7350 call
- Manually close the full structure on 6 May 2026

The notebook has two modes:

1. **Actual realized PnL**, using the real entry and exit option prices from the CME simulator.
2. **Estimated hourly PnL**, using Black-76 option pricing from an hourly AUD/USD futures path.

Important: hourly option PnL is only exact if you paste actual hourly option marks. Otherwise, the model-based hourly PnL is an estimate.


## 1. Imports and settings

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from math import log, sqrt, exp, erf
from zoneinfo import ZoneInfo

pd.set_option("display.float_format", lambda x: f"{x:,.6f}")

SYDNEY = ZoneInfo("Australia/Sydney")
CDT = ZoneInfo("America/Chicago")

CONTRACT_MULTIPLIER = 100_000  # CME AUD option on futures is based on 100,000 AUD


## 2. Trade setup

Replace the sample `entry_price` and `exit_price` values with your actual CME simulator fills.

Prices should be written in AUD/USD option premium points.

Example:

- Premium of 0.0032 means 32 ticks
- Since 0.0001 = $10, 0.0032 = $320 per contract


In [ ]:
# =========================
# USER INPUT SECTION
# =========================

ENTRY_TIME_SYDNEY = pd.Timestamp("2026-05-04 10:30", tz=SYDNEY)

# You only told me the close date, not the exact close time.
# Change this to the real manual exit time from your trading log.
EXIT_TIME_SYDNEY = pd.Timestamp("2026-05-06 15:30", tz=SYDNEY)

# Set this to the actual option expiry used in the simulator.
# For the dashboard, this is editable.
EXPIRY_TIME_SYDNEY = pd.Timestamp("2026-05-08 17:00", tz=SYDNEY)

# Replace entry_price and exit_price with your actual fills from CME.
# Current values are placeholders for the notebook to run.
legs = pd.DataFrame([
    {
        "leg": "Long 0.7200 call",
        "strike": 0.7200,
        "qty": 1,
        "entry_action": "Buy",
        "entry_time_sydney": "2026-05-04 10:30",
        "entry_price": 0.0032,
        "exit_action": "Sell",
        "exit_time_sydney": "2026-05-06 15:30",
        "exit_price": 0.0067,
    },
    {
        "leg": "Short 2x 0.7275 calls",
        "strike": 0.7275,
        "qty": -2,
        "entry_action": "Sell",
        "entry_time_sydney": "2026-05-04 10:31",
        "entry_price": 0.0011,
        "exit_action": "Buy back",
        "exit_time_sydney": "2026-05-06 15:31",
        "exit_price": 0.0019,
    },
    {
        "leg": "Long 0.7350 call",
        "strike": 0.7350,
        "qty": 1,
        "entry_action": "Buy",
        "entry_time_sydney": "2026-05-04 10:32",
        "entry_price": 0.0003,
        "exit_action": "Sell",
        "exit_time_sydney": "2026-05-06 15:32",
        "exit_price": 0.00005,
    },
])

legs["entry_time_sydney"] = pd.to_datetime(legs["entry_time_sydney"]).dt.tz_localize(SYDNEY)
legs["exit_time_sydney"] = pd.to_datetime(legs["exit_time_sydney"]).dt.tz_localize(SYDNEY)

legs["entry_time_cdt"] = legs["entry_time_sydney"].dt.tz_convert(CDT)
legs["exit_time_cdt"] = legs["exit_time_sydney"].dt.tz_convert(CDT)

legs


## 3. Realized PnL from actual entry and exit prices

This section calculates the true final PnL once you replace the placeholders with your real CME simulator prices.


In [ ]:
def calculate_realized_pnl(legs_df: pd.DataFrame, multiplier: float = CONTRACT_MULTIPLIER) -> pd.DataFrame:
    out = legs_df.copy()
    out["premium_paid_at_entry_usd"] = np.where(
        out["qty"] > 0,
        out["qty"] * out["entry_price"] * multiplier,
        0.0
    )
    out["premium_received_at_entry_usd"] = np.where(
        out["qty"] < 0,
        abs(out["qty"]) * out["entry_price"] * multiplier,
        0.0
    )
    out["pnl_usd"] = out["qty"] * (out["exit_price"] - out["entry_price"]) * multiplier
    out["pnl_per_contract_usd"] = (out["exit_price"] - out["entry_price"]) * multiplier
    return out

realized = calculate_realized_pnl(legs)

summary = pd.DataFrame({
    "Metric": [
        "Total entry debit, USD",
        "Total realized PnL, USD",
        "Gross premium paid, USD",
        "Gross premium received, USD",
        "Return on net debit"
    ],
    "Value": [
        (legs["qty"] * legs["entry_price"] * CONTRACT_MULTIPLIER).sum(),
        realized["pnl_usd"].sum(),
        realized["premium_paid_at_entry_usd"].sum(),
        realized["premium_received_at_entry_usd"].sum(),
        realized["pnl_usd"].sum() / abs((legs["qty"] * legs["entry_price"] * CONTRACT_MULTIPLIER).sum())
    ]
})

display(realized[[
    "leg", "qty", "strike", "entry_price", "exit_price",
    "premium_paid_at_entry_usd", "premium_received_at_entry_usd", "pnl_usd"
]])

display(summary)


## 4. Trading log table

This creates a clean table you can paste into the assignment report.

Replace prices with actual simulator fills before using it.


In [ ]:
entry_rows = []
exit_rows = []

for _, row in legs.iterrows():
    entry_rows.append({
        "Date / time Sydney": row["entry_time_sydney"].strftime("%d %b %Y, %H:%M"),
        "Date / time CDT": row["entry_time_cdt"].strftime("%d %b %Y, %H:%M"),
        "Action": "Entered",
        "Instrument": "CME AUD/USD option on 6A futures",
        "Position": f'{row["entry_action"]} {abs(row["qty"])}x {row["strike"]:.4f} call',
        "Price": row["entry_price"],
    })

    exit_rows.append({
        "Date / time Sydney": row["exit_time_sydney"].strftime("%d %b %Y, %H:%M"),
        "Date / time CDT": row["exit_time_cdt"].strftime("%d %b %Y, %H:%M"),
        "Action": "Manually closed",
        "Instrument": "CME AUD/USD option on 6A futures",
        "Position": f'{row["exit_action"]} {abs(row["qty"])}x {row["strike"]:.4f} call',
        "Price": row["exit_price"],
    })

trading_log = pd.DataFrame(entry_rows + exit_rows)
trading_log


## 5. Butterfly payoff at expiry

This shows why the butterfly was chosen.

The position benefits most if AUD/USD ends near the middle strike, **0.7275**.

The risk is limited to the net debit paid at entry.


In [ ]:
def call_payoff(S, K):
    return np.maximum(S - K, 0.0)

lower_K = 0.7200
middle_K = 0.7275
upper_K = 0.7350

net_debit_points = (legs["qty"] * legs["entry_price"]).sum()
net_debit_usd = net_debit_points * CONTRACT_MULTIPLIER

S_grid = np.linspace(0.7050, 0.7450, 400)

payoff_points = (
    call_payoff(S_grid, lower_K)
    - 2 * call_payoff(S_grid, middle_K)
    + call_payoff(S_grid, upper_K)
    - net_debit_points
)

payoff_usd = payoff_points * CONTRACT_MULTIPLIER

plt.figure(figsize=(11, 6))
plt.plot(S_grid, payoff_usd, linewidth=2)
plt.axhline(0, linewidth=1)
plt.axvline(lower_K, linestyle="--", linewidth=1, label="Lower strike 0.7200")
plt.axvline(middle_K, linestyle="--", linewidth=1, label="Max profit strike 0.7275")
plt.axvline(upper_K, linestyle="--", linewidth=1, label="Upper strike 0.7350")
plt.title("AUD/USD Call Butterfly Payoff at Expiry")
plt.xlabel("AUD/USD futures price at expiry")
plt.ylabel("PnL, USD")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print(f"Net debit paid: {net_debit_points:.6f} points = ${net_debit_usd:,.2f}")
print(f"Maximum intrinsic value before premium: {(middle_K - lower_K) * CONTRACT_MULTIPLIER:,.2f} USD")
print(f"Maximum profit after premium: ${((middle_K - lower_K) * CONTRACT_MULTIPLIER - net_debit_usd):,.2f}")
print(f"Maximum loss: ${net_debit_usd:,.2f}")
print(f"Lower breakeven: {lower_K + net_debit_points:.6f}")
print(f"Upper breakeven: {upper_K - net_debit_points:.6f}")


## 6. Estimated hourly mark-to-market PnL

For exact hourly PnL, paste actual hourly option mid-prices from CME.

If you do not have them, this notebook estimates hourly option marks using a Black-76 model from an AUD/USD futures path.

You can replace the demo underlying path with a CSV named `audusd_hourly.csv` with columns:

- `datetime_sydney`
- `audusd_futures`

Example:

```text
datetime_sydney,audusd_futures
2026-05-04 10:30,0.7200
2026-05-04 11:30,0.7203
...
```


In [ ]:
# =========================
# HOURLY DATA INPUT
# =========================

USE_DEMO_UNDERLYING_PATH = True

if USE_DEMO_UNDERLYING_PATH:
    # Demo path only. Replace with your actual hourly AUD/USD futures data for real analysis.
    times = pd.date_range(ENTRY_TIME_SYDNEY, EXIT_TIME_SYDNEY, freq="1H")

    anchors = pd.Series(
        data=[0.7200, 0.7215, 0.7240, 0.7268],
        index=pd.to_datetime([
            "2026-05-04 10:30",
            "2026-05-05 09:30",
            "2026-05-05 15:30",
            "2026-05-06 15:30",
        ]).tz_localize(SYDNEY)
    )

    base = anchors.reindex(times.union(anchors.index)).sort_index().interpolate(method="time").reindex(times)
    np.random.seed(42)
    noise = np.random.normal(0, 0.00025, len(base)).cumsum() * 0.15
    audusd = base.values + noise

    underlying = pd.DataFrame({
        "datetime_sydney": times,
        "audusd_futures": audusd
    })
else:
    underlying = pd.read_csv("audusd_hourly.csv")
    underlying["datetime_sydney"] = pd.to_datetime(underlying["datetime_sydney"]).dt.tz_localize(SYDNEY)

underlying.head()


In [ ]:
def norm_cdf(x):
    return 0.5 * (1.0 + erf(x / sqrt(2.0)))

def black76_call(F, K, T, sigma, r=0.0):
    # Short-dated event model.
    # If option is expired or volatility is zero, return intrinsic value.
    if T <= 0 or sigma <= 0:
        return max(F - K, 0.0)
    d1 = (log(F / K) + 0.5 * sigma**2 * T) / (sigma * sqrt(T))
    d2 = d1 - sigma * sqrt(T)
    return exp(-r * T) * (F * norm_cdf(d1) - K * norm_cdf(d2))

def annualized_time_to_expiry(current_time, expiry_time):
    seconds = max((expiry_time - current_time).total_seconds(), 0)
    return seconds / (365.0 * 24 * 60 * 60)

def event_volatility_assumption(t):
    # Editable assumption:
    # Higher implied volatility before the RBA, then volatility compression after the event.
    rba_time = pd.Timestamp("2026-05-05 14:30", tz=SYDNEY)
    if t < rba_time:
        return 0.105
    return 0.085

mtm_rows = []

for _, obs in underlying.iterrows():
    t = obs["datetime_sydney"]
    F = obs["audusd_futures"]
    T = annualized_time_to_expiry(t, EXPIRY_TIME_SYDNEY)
    sigma = event_volatility_assumption(t)

    row = {
        "datetime_sydney": t,
        "audusd_futures": F,
        "implied_vol_assumption": sigma,
    }

    total_pnl = 0.0

    for _, leg in legs.iterrows():
        mark = black76_call(F, leg["strike"], T, sigma)
        pnl = leg["qty"] * (mark - leg["entry_price"]) * CONTRACT_MULTIPLIER

        row[f'mark_{leg["strike"]:.4f}'] = mark
        row[f'pnl_{leg["strike"]:.4f}'] = pnl
        total_pnl += pnl

    row["estimated_total_pnl_usd"] = total_pnl
    mtm_rows.append(row)

mtm = pd.DataFrame(mtm_rows)
mtm.tail()


## 7. Dashboard charts

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(mtm["datetime_sydney"], mtm["audusd_futures"], linewidth=2)
plt.axhline(0.7200, linestyle="--", linewidth=1, label="0.7200 lower strike")
plt.axhline(0.7275, linestyle="--", linewidth=1, label="0.7275 max profit strike")
plt.axhline(0.7350, linestyle="--", linewidth=1, label="0.7350 upper strike")
plt.title("AUD/USD Futures Path During the Trade")
plt.xlabel("Sydney time")
plt.ylabel("AUD/USD futures price")
plt.legend()
plt.grid(True, alpha=0.3)
plt.xticks(rotation=30)
plt.show()

plt.figure(figsize=(12, 5))
plt.plot(mtm["datetime_sydney"], mtm["estimated_total_pnl_usd"], linewidth=2)
plt.axhline(0, linewidth=1)
plt.title("Estimated Hourly Mark-to-Market PnL")
plt.xlabel("Sydney time")
plt.ylabel("Estimated PnL, USD")
plt.grid(True, alpha=0.3)
plt.xticks(rotation=30)
plt.show()

leg_pnl = realized[["leg", "pnl_usd"]].copy()
plt.figure(figsize=(10, 5))
plt.bar(leg_pnl["leg"], leg_pnl["pnl_usd"])
plt.axhline(0, linewidth=1)
plt.title("Realized PnL by Option Leg")
plt.xlabel("Option leg")
plt.ylabel("Realized PnL, USD")
plt.grid(True, axis="y", alpha=0.3)
plt.xticks(rotation=20)
plt.show()


## 8. Final performance summary

In [ ]:
total_pnl = realized["pnl_usd"].sum()
net_debit = (legs["qty"] * legs["entry_price"] * CONTRACT_MULTIPLIER).sum()
max_intrinsic = (middle_K - lower_K) * CONTRACT_MULTIPLIER
max_profit = max_intrinsic - net_debit
max_loss = net_debit

performance_summary = pd.DataFrame({
    "Metric": [
        "Strategy",
        "Entry date",
        "Exit date",
        "Net debit paid, USD",
        "Maximum possible loss, USD",
        "Maximum possible profit at expiry, USD",
        "Realized PnL, USD",
        "Return on net debit",
        "Best estimated hourly PnL, USD",
        "Worst estimated hourly PnL, USD"
    ],
    "Value": [
        "Long 0.7200 / 0.7275 / 0.7350 AUD/USD call butterfly",
        ENTRY_TIME_SYDNEY.strftime("%d %b %Y, %H:%M Sydney"),
        EXIT_TIME_SYDNEY.strftime("%d %b %Y, %H:%M Sydney"),
        f"${net_debit:,.2f}",
        f"${max_loss:,.2f}",
        f"${max_profit:,.2f}",
        f"${total_pnl:,.2f}",
        f"{total_pnl / abs(net_debit):.2%}",
        f"${mtm['estimated_total_pnl_usd'].max():,.2f}",
        f"${mtm['estimated_total_pnl_usd'].min():,.2f}"
    ]
})

performance_summary


## 9. Report wording you can adapt

The following wording is generated from the trade structure.

Replace the PnL numbers after adding your real fills.


In [ ]:
report_text = f'''
We entered a long AUD/USD call butterfly using CME Australian Dollar options on futures ahead of the RBA policy decision. 
The structure consisted of buying one 0.7200 call, selling two 0.7275 calls, and buying one 0.7350 call. 
This created a defined-risk bullish payoff designed to profit from a moderate AUD/USD appreciation if the RBA delivered a hawkish surprise.

The butterfly was chosen instead of an outright long futures position because our view was not that AUD/USD would rise without limit. 
Rather, we expected a controlled repricing toward the 0.7275 area. 
The maximum loss was limited to the net premium paid, while the maximum payoff occurred if AUD/USD moved close to the middle strike. 
We manually exited the full structure on 6 May 2026, after the market had time to digest the RBA decision, by reversing all three option legs.

Using the current input prices, the strategy generated a realized PnL of ${total_pnl:,.2f}. 
This number should be updated with the actual simulator fills before submission.
'''
print(report_text)
